# 04 - QC summary

This notebook intends to summarize the various entries that occur during the thematic and geometric QC/validation similar to those reported in the publication.
A few convince functions from the `utils` module help with that.

## Thematic validation:
uses the `qc_reviewed`, `qc_flag`, `qc_crop_main`, and `qc_crop_sec` columns produced by the `fieldqc.run_qc` tool to compute accuracy metrics and relabel counts.

### Accuracy definitions

| metric | formula | definition                                                                                                                                                                      |
|---|--------------|---------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `field_accuracy` | `n_correct / n_assessable`| all labels of the field were confirmed correct.                                                                                                  |
| `main_accuracy`  | `(n_assessable - n_main_changed) / n_assessable`| main crop was correct.                                                                                                     |
| `sec_accuracy`   | `(n_assessed_with_sec - n_sec_changed) / n_assessed_with_sec`| secondary crop was correct, denominator restricted to assessable fields that originally had a secondary crop. |

`skip` rows are excluded from every accuracy denominator and reported separately. Fields where a secondary crop was added during QC are reported as `n_sec_added` but are not included in the secondary-accuracy denominator.

## Geometric validation

The polygon geometry is reviewed manually in QGIS against the SPOT 6/7 satellite imagery and tracked in `qc_shape_*` columns of the `_QC.gpkg` files:
uses the `qc_shape_reviewed`, `qc_shape_flag`, and `qc_shape_offset_m` columns to compute relevant metrics.


In [8]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

from utils import compute_qc_metrics, print_qc_summary, compute_shape_metrics, print_shape_summary

## Load reviewed GeoPackage

Reads the QC output produced from `03_qc.ipynb` (suffix `_QC.gpkg`).


In [ ]:
DATA_DIR = Path("../data")
CAMPAIGN = "01_LRS23"
CODE     = "LRS23"

qc_path = DATA_DIR / CAMPAIGN / f"CropHype-Fields-Kenya_{CODE}_QC.gpkg"
gdf = gpd.read_file(qc_path)
print(f"{CODE}: {len(gdf)} fields, {int(gdf['qc_reviewed'].notna().sum())} reviewed")

## Thematic validation summary

In [ ]:
m = compute_qc_metrics(gdf, name=CODE)
print_qc_summary(m) # convenience function

### Per-crop accuracy

In [ ]:
per_crop = m["per_crop_df"].copy()
per_crop["accuracy"] = per_crop["accuracy"].map(lambda v: f"{100 * v:.1f}%")
per_crop

### Main-crop relabel pairs

Rows where the reviewer changed the main crop label, aggregated as `from -> to` counts. Empty when no relabelings occurred.

In [ ]:
m["main_changes_df"]

## Geometric validation summary

`compute_shape_metrics` returns a dict with the shape sample counts, accuracy and offset statistics; `print_shape_summary` formats it.

In [ ]:
s = compute_shape_metrics(gdf, name=CODE)
print_shape_summary(s)

## Cross-campaign comparison

`compute_qc_metrics` and `compute_shape_metrics` each return a dict per campaign. Aggregate over campaigns for an overview similar to publication.

In [ ]:
CAMPAIGNS = [
    ("01_LRS23", "LRS23"),
    ("02_SRS23", "SRS23"),
    ("03_LRS24", "LRS24"),
    ("04_LRS25", "LRS25"),
    ("05_SRS25", "SRS25"),
]

rows = []
for camp, c in CAMPAIGNS:
    g  = gpd.read_file(DATA_DIR / camp / f"CropHype-Fields-Kenya_{c}_QC.gpkg")
    mi = compute_qc_metrics(g, name=c)
    si = compute_shape_metrics(g, name=c)
    rows.append({
        "Campaign":         mi["name"],
        "Total":            mi["n_total"],
        "Intercrop":        mi["n_intercrop"],
        "Reviewed":         mi["n_reviewed"],
        "Skipped":          mi["n_skipped"],
        "Assessable":       mi["n_assessable"],
        "Field acc.":       f"{100 * mi['field_accuracy']:.1f}%",
        "Main acc.":        f"{100 * mi['main_accuracy']:.1f}%",
        "Sec acc.":         f"{100 * mi['sec_accuracy']:.1f}%",
        "Shape reviewed":   si["n_shape_reviewed"],
        "Shape skip":       f"{si['n_shape_skipped']} ({100*si['pct_shape_skipped']:.1f}%)",
        "Shape mismatch":   f"{si['n_shape_mismatch']} ({100*si['pct_shape_mismatch']:.1f}%)",
        "Offset > 0":       f"{si['n_with_offset']} ({100*si['pct_with_offset']:.1f}%)",
        "Offset mean (m)":  f"{si['mean_offset_m']:.2f}" if pd.notna(si['mean_offset_m']) else "N/A",
        "Offset sd (m)":    f"{si['std_offset_m']:.2f}"  if pd.notna(si['std_offset_m'])  else "N/A",
    })

summary = pd.DataFrame(rows).set_index("Campaign")
summary